# 0 - Provision LightGBM serving system node

Use this notebook in the Chameleon Jupyter environment to provision the VM for LightGBM serving experiments.

This path uses the training teammate's model from MLflow instead of the placeholder ONNX artifact.

Defaults in this workflow:
- tracking URI: `http://129.114.24.226:30500`
- model path: `/models/smartqueue_lgbm.txt` (loaded from training artifacts)
- default machine type: `m1.medium`


In [ ]:
# runs in Chameleon Jupyter environment
from chi import server, context, lease, network
import chi, os, datetime

context.version = "1.0"
context.choose_project()
context.choose_site(default="KVM@TACC")

username = os.getenv("USER")
PROJECT_SUFFIX = "proj13"
LEASE_NAME = f"lease-serve-lightgbm-{username}-{PROJECT_SUFFIX}"
SERVER_NAME = f"node-serve-lightgbm-{username}-{PROJECT_SUFFIX}"
FLAVOR_NAME = "m1.medium"  # change this if you want to compare machine types
IMAGE_NAME = "CC-Ubuntu24.04"


In [ ]:
# runs in Chameleon Jupyter environment
l = lease.Lease(LEASE_NAME, duration=datetime.timedelta(hours=8))
l.add_flavor_reservation(id=chi.server.get_flavor_id(FLAVOR_NAME), amount=1)
l.submit(idempotent=True)
l.show()


In [ ]:
# runs in Chameleon Jupyter environment
s = server.Server(
    SERVER_NAME,
    image_name=IMAGE_NAME,
    flavor_name=l.get_reserved_flavors()[0].name,
)
s.submit(idempotent=True)


In [ ]:
# runs in Chameleon Jupyter environment
s.associate_floating_ip()
s.refresh()
s.check_connectivity()


In [ ]:
# runs in Chameleon Jupyter environment
security_groups = [
    {"name": "allow-ssh", "port": 22, "description": "Enable SSH traffic on TCP port 22"},
    {"name": "allow-8888", "port": 8888, "description": "Enable TCP port 8888 (used by Jupyter)"},
    {"name": "allow-8000", "port": 8000, "description": "Enable TCP port 8000 (serving API)"},
]

for sg in security_groups:
    secgroup = network.SecurityGroup({
        "name": sg["name"],
        "description": sg["description"],
    })
    secgroup.add_rule(direction="ingress", protocol="tcp", port=sg["port"])
    secgroup.submit(idempotent=True)
    s.add_security_group(sg["name"])

print(f"updated security groups: {[sg['name'] for sg in security_groups]}")
s.refresh()
s.show(type="widget")


In [ ]:
# runs in Chameleon Jupyter environment
s.execute("curl -sSL https://get.docker.com/ | sudo sh")
s.execute("sudo groupadd -f docker; sudo usermod -aG docker $USER")
s.execute("sudo apt-get install -y docker-compose-plugin")


In [ ]:
# runs in Chameleon Jupyter environment
REPO_URL = "https://github.com/yanghao13111/SmartQueue.git"
s.execute(f"git clone {REPO_URL} ~/smartqueue")


Open an SSH session from your local terminal:

```bash
ssh -i ~/.ssh/id_rsa_chameleon cc@A.B.C.D
```

Replace `A.B.C.D` with the floating IP shown above.


Start the LightGBM baseline service from the VM host shell:

```bash
cd ~/smartqueue/serving/docker
MODEL_DIR=~/smartqueue/training/artifacts MODEL_VERSION=lightgbm_v4_baseline UVICORN_WORKERS=1 \
MLFLOW_TRACKING_URI=http://129.114.24.226:30500 \
docker compose -f docker-compose-lightgbm.yaml up -d --build --force-recreate fastapi_lgbm jupyter
```

Health check:

```bash
curl http://localhost:8000/health
```


Run the saved evaluation for the baseline option:

```bash
cd ~/smartqueue/serving/docker
MODEL_DIR=~/smartqueue/training/artifacts MODEL_VERSION=lightgbm_v4_baseline UVICORN_WORKERS=1 \
MLFLOW_TRACKING_URI=http://129.114.24.226:30500 \
docker compose --profile eval -f docker-compose-lightgbm.yaml run --rm eval sh run_evaluation.sh lightgbm_v4_baseline
```

Then rerun with multi-worker FastAPI:

```bash
cd ~/smartqueue/serving/docker
MODEL_DIR=~/smartqueue/training/artifacts MODEL_VERSION=lightgbm_v4_concurrent UVICORN_WORKERS=4 \
MLFLOW_TRACKING_URI=http://129.114.24.226:30500 \
docker compose -f docker-compose-lightgbm.yaml up -d --build --force-recreate fastapi_lgbm

MODEL_DIR=~/smartqueue/training/artifacts MODEL_VERSION=lightgbm_v4_concurrent UVICORN_WORKERS=4 \
MLFLOW_TRACKING_URI=http://129.114.24.226:30500 \
docker compose --profile eval -f docker-compose-lightgbm.yaml run --rm eval sh run_evaluation.sh lightgbm_v4_concurrent
```

If you want a machine-type comparison, change `FLAVOR_NAME` in this notebook and use a distinct option name such as `lightgbm_mlflow_concurrent_m1_large`.


After the service is running, open and run:

- `8_lightgbm_mlflow_serving.ipynb`


In [ ]:
# runs in Chameleon Jupyter environment
# delete the VM when finished to keep costs low
s.delete()
